# 115. 미성형8개 누락·정상 중첩 진단
113/114 실행 후 설명용 감사. 평가 정답을 피처·임계값 선택에 다시 쓰지 않는다. 원값9개 두자리 일치는 현재 센서 일부의 중첩일 뿐 전체 피처가 같다는 의미 아님. 관측구간 정상 순위는 사후 설명용이다.


In [1]:
from pathlib import Path
import pandas as pd,numpy as np
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
p=pd.read_csv(t/'rg3_shortshot_specialist_predictions.csv',parse_dates=['TimeStamp'])
contrast=pd.read_csv(t/'rg3_shortshot_case_contrast_predictions.csv',parse_dates=['TimeStamp'])
primary=p[p.model.eq('primary_inner_selected')].set_index('TimeStamp')
candidate=contrast[contrast.model.eq('raw6_k1')&contrast.rule.eq('normal99')].set_index('TimeStamp')
cols=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Filling_Time','Injection_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Speed','Barrel_Temperature_5','Mold_Temperature_3']
normal=d[d.PassOrFail.eq(0)]; bad=d[d.PassOrFail.eq(1)]
rows=[]
for _,a in bad.iterrows():
 same=normal[normal.comparison_block.eq(a.comparison_block)]
 percentile={v:float((same[v]<a[v]).mean()+.5*(same[v]==a[v]).mean()) for v in cols}
 exact=normal[normal[cols].round(2).eq(pd.to_numeric(a[cols]).round(2),axis=1).all(axis=1)]
 z=primary.loc[a.TimeStamp]; other=primary[primary.block.eq(a.comparison_block)&primary.actual.eq(0)]
 ahead=int((other.score>=z.score).sum())
 q=candidate.loc[a.TimeStamp]
 rows.append(dict(TimeStamp=a.TimeStamp,block=a.comparison_block,primary_prediction=int(z.prediction),primary_score=float(z.score),threshold=float(z.threshold),same_block_normals_scored_at_least_as_high=ahead,case_contrast_prediction=int(q.prediction),rounded9_normal_matches=len(exact),matching_normal_timestamps=';'.join(exact.TimeStamp.astype(str)),**{v+'_sameblock_midrank':val for v,val in percentile.items()}))
r=pd.DataFrame(rows)
r.to_csv(t/'rg3_shortshot_failure_all8_audit.csv',index=False,encoding='utf-8-sig')
choices=pd.read_csv(t/'rg3_shortshot_specialist_inner_choices.csv')
print('INNER CHOICES',choices.to_string(index=False))
print('ALL8',r[['TimeStamp','primary_prediction','same_block_normals_scored_at_least_as_high','case_contrast_prediction','rounded9_normal_matches','matching_normal_timestamps']].to_string(index=False))
# Diagnostic only: no thresholds or models selected from this report.
summary=pd.DataFrame({'feature':cols,'shortshot_min':bad[cols].min().to_numpy(),'shortshot_max':bad[cols].max().to_numpy(),'normal_p05':normal[cols].quantile(.05).to_numpy(),'normal_p95':normal[cols].quantile(.95).to_numpy()})
summary.to_csv(t/'rg3_shortshot_sensor_overlap_audit.csv',index=False,encoding='utf-8-sig')
print(summary.to_string(index=False))


INNER CHOICES               block                     selected  inner_F1  threshold
20201021_001626_RG3 sequence6+conditional_phase6  0.200000   0.625149
20201022_002405_RG3                 phase_proxy6  0.200000   0.704663
20201022_044026_RG3      raw6+conditional_phase6  0.200000   0.542587
20201023_010128_RG3           conditional_phase6  0.071429   0.635732
20201023_044148_RG3                    sequence6  0.285714   0.656944
ALL8           TimeStamp  primary_prediction  same_block_normals_scored_at_least_as_high  case_contrast_prediction  rounded9_normal_matches matching_normal_timestamps
2020-10-22 01:09:23                   0                                          59                         0                        0                           
2020-10-22 04:48:39                   0                                          93                         0                        0                           
2020-10-22 05:42:13                   0                                    